# Face Verification — MobileNetV3-Large + Batch Hard Triplet Loss + Image 112×112 | Split 80/10/10
## Notebook Complet — Kaggle GPU | Image 112×112 | Split 80/10/10

### Architecture & Entraînement
- **Modèle** : MobileNetV3-Large + Embedding 512D (Face Verification)
- **Loss** : Batch Hard Triplet Loss (margin = 0.5)
- **Optimizer** : AdamW (LR tête = 1e-3, backbone = 1e-4 en fine-tuning)
- **Scheduler** : CosineAnnealingLR
- **Batch** : 108 (P=18 personnes × K=6 images/personne)
- **Epochs** : 40 (early stopping patience=10, best ≈ epoch 30)
- **Dataset** : 745 personnes | 11 321 images ≈ 9 057 train / 1 132 val / 1 132 test
- **Split** : 80 / 10 / 10 (stratifié par personne)
- **Test protocol** : 9 000 paires (6 000 positifs | 3 000 négatifs)

## 1a. Installation PyTorch (Kaggle P100 / SM 6.0)

In [1]:
import subprocess, sys

print("=" * 70)
print("PYTORCH INSTALLER FOR KAGGLE P100 (SM 6.0)")
print("=" * 70)

subprocess.run([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "torch", "torchvision", "torchaudio"
], check=False)

print("\nInstalling torch==2.2.0+cu118")
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "torch==2.2.0+cu118", "torchvision==0.17.0+cu118", "torchaudio==2.2.0+cu118",
    "--index-url", "https://download.pytorch.org/whl/cu118", "-q"
], check=True)

print("\nPinning numpy==1.26.4")
subprocess.run([
    sys.executable, "-m", "pip", "install", "numpy==1.26.4", "-q"
], check=True)

print("\nInstalling timm, tqdm, scikit-learn")
subprocess.run([
    sys.executable, "-m", "pip", "install", "timm", "tqdm", "scikit-learn", "-q"
], check=True)

print("\nAll installations complete.")

PYTORCH INSTALLER FOR KAGGLE P100 (SM 6.0)
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128

Installing torch==2.2.0+cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.6/811.6 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 99.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 75.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 45.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 103.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you 


Installing timm, tqdm, scikit-learn

All installations complete.


## 1b. NumPy Sanity Check

In [2]:
import numpy as np
print(f"NumPy version: {np.__version__}")
assert np.__version__.startswith("1."), f"Expected numpy 1.x, got {np.__version__} -- restart kernel."
print("NumPy version OK.")

NumPy version: 2.0.2


AssertionError: Expected numpy 1.x, got 2.0.2 -- restart kernel.

## 2. Import des Librairies

In [ ]:
import os, gc, json, re, random
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import torchvision
from torchvision import transforms
import timm
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.manifold import TSNE

try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("UMAP not available, t-SNE will be used instead")

from tqdm import tqdm
from scipy.stats import norm as scipy_norm
from pathlib import Path
from collections import defaultdict

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("=" * 70)
print("GPU INFORMATION")
print("=" * 70)
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA version   : {torch.version.cuda}")
else:
    print("No GPU -- CPU will be used (VERY SLOW)")
print("=" * 70)
print("All imports loaded!")

## 3. Configuration

In [ ]:
import os, torch, numpy as np

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TORCH_CUDA_DSA']       = '1'

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

class Config:
    # ── Dataset ───────────────────────────────────────────────────────────────
    USE_SUBSET        = False
    SUBSET_PERCENTAGE = 0.2

    # ── Image ─────────────────────────────────────────────────────────────────
    image_size        = 112

    # ── Split 80 / 10 / 10 ────────────────────────────────────────────────────
    train_ratio       = 0.80
    val_ratio         = 0.10
    test_ratio        = 0.10   # Le reste va au test

    # ── Triplet Batch Sampler (P personnes × K images) ────────────────────────
    P                 = 18     # personnes par batch
    K                 = 6      # images par personne  →  batch_size = P*K = 108
    batch_size        = P * K  # 108

    # ── Training ──────────────────────────────────────────────────────────────
    num_epochs        = 40
    seed              = 42

    # ── Model ─────────────────────────────────────────────────────────────────
    embedding_dim     = 512
    model_name        = "mobilenetv3_large_100"

    # ── Batch Hard Triplet Loss ────────────────────────────────────────────────
    triplet_margin    = 0.5

    # ── Optimizer (différentiel LR) ────────────────────────────────────────────
    lr_head           = 1e-3   # couche d'embedding
    lr_backbone       = 1e-4   # backbone (fine-tuning)
    weight_decay      = 1e-4

    # ── Early stopping ────────────────────────────────────────────────────────
    early_stopping_patience  = 10
    early_stopping_min_delta = 0.0001

    # ── Test protocol ─────────────────────────────────────────────────────────
    test_pos_pairs    = 6000
    test_neg_pairs    = 3000   # total = 9 000 paires

    # ── Output ────────────────────────────────────────────────────────────────
    output_dir        = '/kaggle/working'

    # Sera défini après scan du dataset
    num_classes       = None

torch.manual_seed(Config.seed)
np.random.seed(Config.seed)
random.seed(Config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(Config.seed)

print("\nCONFIG — MobileNetV3 + Batch Hard Triplet Loss")
print("=" * 70)
print(f"  Image size     : {Config.image_size}×{Config.image_size}")
print(f"  Split          : {int(Config.train_ratio*100)}/{int(Config.val_ratio*100)}/{int(Config.test_ratio*100)}")
print(f"  Batch (P×K)    : {Config.P}×{Config.K} = {Config.batch_size}")
print(f"  Epochs         : {Config.num_epochs}")
print(f"  Embedding dim  : {Config.embedding_dim}")
print(f"  Triplet margin : {Config.triplet_margin}")
print(f"  LR head/backbone : {Config.lr_head}/{Config.lr_backbone}")
print(f"  Test pairs     : {Config.test_pos_pairs} pos + {Config.test_neg_pairs} neg")
print(f"  Output dir     : {Config.output_dir}")
print("=" * 70)

## 4. Chargement du Dataset Kaggle

Détection automatique. Tous les sous-dossiers contenant des images sont acceptés.

In [ ]:
def _auto_detect_dataset(kaggle_input='/kaggle/input/datasets/chaimaeelmounjali/maskedfacedataset/GLOBAL_DATASET'):
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

    def has_images(d):
        try:
            if any(os.path.splitext(f)[1].lower() in exts for f in os.listdir(d)):
                return True
            for sub in os.listdir(d):
                sub_p = os.path.join(d, sub)
                if os.path.isdir(sub_p):
                    if any(os.path.splitext(f)[1].lower() in exts for f in os.listdir(sub_p)):
                        return True
        except (PermissionError, NotADirectoryError):
            pass
        return False

    candidates = []
    for root, dirs, files in os.walk(kaggle_input):
        depth = root.replace(kaggle_input, '').count(os.sep)
        if depth > 2:
            dirs.clear()
            continue
        subdirs_with_images = [d for d in dirs if has_images(os.path.join(root, d))]
        if len(subdirs_with_images) >= 2:
            candidates.append((root, len(subdirs_with_images)))
    if not candidates:
        return None
    candidates.sort(key=lambda x: -x[1])
    return candidates[0][0]

_manual_path = None  # Mettez ici votre chemin si l'auto-détection échoue

if _manual_path and os.path.exists(_manual_path):
    dataset_path = _manual_path
    print(f"[INFO] Chemin manuel : {dataset_path}")
else:
    dataset_path = _auto_detect_dataset()
    if dataset_path is None:
        print("\n[ERREUR] Aucun dataset trouvé automatiquement.")
        for entry in sorted(os.listdir('/kaggle/input')):
            print(f"  /kaggle/input/{entry}")
        raise FileNotFoundError("Dataset introuvable. Fixez _manual_path.")
    print(f"[AUTO] Dataset détecté : {dataset_path}")

def _count_images(d):
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    try:
        return sum(1 for f in os.listdir(d) if os.path.splitext(f)[1].lower() in exts)
    except PermissionError:
        return 0

def _has_images(p):
    if _count_images(p) > 0:
        return True
    try:
        for sub in os.listdir(p):
            sp = os.path.join(p, sub)
            if os.path.isdir(sp) and _count_images(sp) > 0:
                return True
    except PermissionError:
        pass
    return False

person_dirs = sorted([
    d for d in os.listdir(dataset_path)
    if os.path.isdir(os.path.join(dataset_path, d))
    and _has_images(os.path.join(dataset_path, d))
])

total_images = 0
for d in person_dirs:
    dp = os.path.join(dataset_path, d)
    total_images += _count_images(dp)
    try:
        for sub in os.listdir(dp):
            sp = os.path.join(dp, sub)
            if os.path.isdir(sp):
                total_images += _count_images(sp)
    except PermissionError:
        pass

Config.num_classes = len(person_dirs)

print("=" * 70)
print(f"  Path           : {dataset_path}")
print(f"  Personnes      : {Config.num_classes}")
print(f"  Images totales : {total_images}")
print(f"  Avg/classe     : {total_images // max(Config.num_classes, 1)}")
print("=" * 70)

## 5. Batch Hard Triplet Loss

Pour chaque ancre on sélectionne **le positif le plus difficile** (cosine sim la plus basse avec même identité) et **le négatif le plus difficile** (cosine sim la plus haute avec identité différente). Cela force le modèle à apprendre les cas limites.

In [ ]:
class BatchHardTripletLoss(nn.Module):
    """
    Batch Hard Triplet Loss (Hermans et al. 2017 — In Defense of the Triplet Loss).

    Pour chaque ancre i dans le batch :
      - Hardest positive  : argmax_{j: y_j == y_i} d(i, j)
      - Hardest negative  : argmin_{j: y_j != y_i} d(i, j)

    Distance = 1 - cosine_similarity  (∈ [0, 2])
    Margin   = 0.5  (hinge loss)
    """
    def __init__(self, margin: float = 0.5):
        super().__init__()
        self.margin = margin

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor):
        """
        Args:
            embeddings : (B, D) — L2-normalized embeddings
            labels     : (B,)   — integer class labels
        Returns:
            scalar loss, fraction of non-zero triplets (active fraction)
        """
        # ── Pairwise cosine distance matrix ──────────────────────────────────
        # embeddings déjà normalisés → dot product = cosine similarity
        sim    = torch.mm(embeddings, embeddings.t())          # (B, B)
        dist   = 1.0 - sim                                     # cosine distance

        B = labels.size(0)
        labels_equal = labels.unsqueeze(0) == labels.unsqueeze(1)  # (B, B) bool

        # ── Hardest positive ─────────────────────────────────────────────────
        # Masque : même classe MAIS pas soi-même
        pos_mask = labels_equal & ~torch.eye(B, dtype=torch.bool, device=embeddings.device)
        # Pour les ancres sans positif dans le batch, on met 0 (pas de contribution)
        dist_pos = dist * pos_mask.float()
        hardest_pos = dist_pos.max(dim=1).values                   # (B,)

        # ── Hardest negative ─────────────────────────────────────────────────
        # Masque : classe différente
        neg_mask = ~labels_equal
        # On veut le min → on remplace les positions invalides par +inf
        dist_neg = dist.clone()
        dist_neg[~neg_mask] = float('inf')
        hardest_neg = dist_neg.min(dim=1).values                   # (B,)

        # ── Triplet loss (batch hard) ─────────────────────────────────────────
        triplet_loss = F.relu(hardest_pos - hardest_neg + self.margin)

        # Ignorer les ancres sans positif valide
        valid = pos_mask.any(dim=1)
        if valid.sum() == 0:
            return torch.tensor(0.0, device=embeddings.device, requires_grad=True), 0.0

        loss   = triplet_loss[valid].mean()
        active = (triplet_loss[valid] > 0).float().mean().item()
        return loss, active

print("BatchHardTripletLoss défini.")

## 6. Architecture — MobileNetV3-Large + Embedding 512D

MobileNetV3-Large pré-entraîné sur ImageNet. On retire le classifieur et on greffe une couche d'embedding 512D avec BatchNorm + normalisation L2. L'embedding est adapté à la vérification faciale par similitude cosinus.

In [ ]:
class FaceVerificationModel(nn.Module):
    """
    MobileNetV3-Large backbone + tete d'embedding 512D.

    Forward : retourne des embeddings L2-normalises (compatibles cosine similarity).
    """
    def __init__(self, embedding_dim: int = 512, model_name: str = "mobilenetv3_large_100", image_size: int = 112):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,      # Retire le classifieur
            global_pool='avg',  # GlobalAvgPool integre
        )

        # Infer actual backbone output size (timm variants can differ)
        was_training = self.backbone.training
        self.backbone.eval()
        with torch.no_grad():
            dummy = torch.zeros(1, 3, image_size, image_size)
            backbone_out = self.backbone(dummy).shape[1]
        if was_training:
            self.backbone.train()

        self.embedding_head = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim, bias=False),
            nn.BatchNorm1d(embedding_dim),
        )
        self.embedding_dim = embedding_dim

        # Initialisation Xavier
        nn.init.xavier_normal_(self.embedding_head[0].weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features   = self.backbone(x)              # (B, backbone_out)
        embeddings = self.embedding_head(features) # (B, embedding_dim)
        return F.normalize(embeddings, p=2, dim=1) # L2-norm

    def freeze_backbone(self):
        """Phase 1 : seule la tete est entrainee."""
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        """Phase 2 : fine-tuning complet du backbone."""
        for p in self.backbone.parameters():
            p.requires_grad = True

    def get_param_groups(self, lr_head: float, lr_backbone: float, wd: float):
        """Groupes de parametres pour AdamW avec LR differentiel."""
        return [
            {'params': self.backbone.parameters(),       'lr': lr_backbone, 'weight_decay': wd},
            {'params': self.embedding_head.parameters(), 'lr': lr_head,     'weight_decay': wd},
        ]


model = FaceVerificationModel(
    embedding_dim=Config.embedding_dim,
    model_name=Config.model_name,
    image_size=Config.image_size,
).to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
head_p    = sum(p.numel() for p in model.embedding_head.parameters())
print(f"Backbone          : {Config.model_name} (features={model.embedding_head[0].in_features})")
print(f"Embedding dim     : {Config.embedding_dim}")
print(f"Total params      : {total:,}")
print(f"Trainable params  : {trainable:,}")
print(f"Head params       : {head_p:,}")
print("Model ready.")

## 7. Dataset, Transforms et PK Sampler

### PK Sampler
À chaque batch on tire exactement **P=18 personnes** et **K=6 images** par personne, ce qui garantit la présence de triplets positifs valides dans chaque batch.

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((Config.image_size + 20, Config.image_size + 20)),
    transforms.RandomCrop(Config.image_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),
])

val_transforms = transforms.Compose([
    transforms.Resize((Config.image_size + 20, Config.image_size + 20)),
    transforms.CenterCrop(Config.image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

# ── Generic FaceDataset ───────────────────────────────────────────────────────
def _collect_images(person_dir):
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    images = []
    p = Path(person_dir)
    for f in p.iterdir():
        if f.is_file() and f.suffix.lower() in exts:
            images.append(str(f))
    for sub in p.iterdir():
        if sub.is_dir():
            for f in sub.iterdir():
                if f.is_file() and f.suffix.lower() in exts:
                    images.append(str(f))
    return images

class FaceDataset(Dataset):
    def __init__(self, dataset_path, person_dirs, transform=None):
        self.transform    = transform
        self.classes      = person_dirs
        self.class_to_idx = {c: i for i, c in enumerate(person_dirs)}
        self.idx_to_class = {i: c for i, c in enumerate(person_dirs)}
        self.samples      = []          # list of (path, label)
        self.labels       = []          # aligned list of labels
        self.label_to_indices = defaultdict(list)

        for cls_name in person_dirs:
            cls_idx  = self.class_to_idx[cls_name]
            cls_path = os.path.join(dataset_path, cls_name)
            imgs     = _collect_images(cls_path)
            for img_path in imgs:
                self.samples.append((img_path, cls_idx))
                self.labels.append(cls_idx)
                self.label_to_indices[cls_idx].append(len(self.samples) - 1)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

# ── PK Batch Sampler ─────────────────────────────────────────────────────────
class PKBatchSampler(Sampler):
    """
    À chaque itération tire P personnes et K images par personne.
    Garantit la présence de positifs dans chaque batch.
    """
    def __init__(self, dataset: FaceDataset, P: int, K: int):
        self.label_to_indices = {
            lbl: list(idxs)
            for lbl, idxs in dataset.label_to_indices.items()
            if len(idxs) >= 2   # au moins 2 images par personne
        }
        self.labels    = list(self.label_to_indices.keys())
        self.P         = P
        self.K         = K
        self.n_batches = max(1, len(self.labels) // P)

    def __iter__(self):
        for _ in range(self.n_batches):
            batch_labels = random.sample(self.labels, min(self.P, len(self.labels)))
            batch_indices = []
            for lbl in batch_labels:
                idxs = self.label_to_indices[lbl]
                chosen = random.choices(idxs, k=self.K) if len(idxs) < self.K else random.sample(idxs, self.K)
                batch_indices.extend(chosen)
            yield batch_indices

    def __len__(self):
        return self.n_batches

print("FaceDataset, PKBatchSampler définis.")

## 8. Split 80 / 10 / 10 (stratifié par personne)

Chaque personne est représentée dans les 3 splits. Les indices sont triés par classe pour la reproductibilité.

In [ ]:
from sklearn.model_selection import train_test_split

# Dataset complet (pour récupérer les indices par classe)
full_dataset = FaceDataset(dataset_path, person_dirs, transform=None)
print(f"Dataset total : {len(full_dataset)} images, {Config.num_classes} classes")

# ── Split stratifié par personne ──────────────────────────────────────────────
train_idx, val_idx, test_idx = [], [], []

rng = np.random.default_rng(Config.seed)

for cls_idx, indices in full_dataset.label_to_indices.items():
    idxs = list(indices)
    rng.shuffle(idxs)
    n     = len(idxs)
    n_val  = max(1, int(n * Config.val_ratio))
    n_test = max(1, int(n * Config.test_ratio))
    n_train = n - n_val - n_test

    if n_train < 1:
        # Trop peu d'images : tout en train
        train_idx.extend(idxs)
        continue

    train_idx.extend(idxs[:n_train])
    val_idx.extend(idxs[n_train:n_train + n_val])
    test_idx.extend(idxs[n_train + n_val:])

print(f"Split 80/10/10 stratifié :")
print(f"  Train  : {len(train_idx):,} images  ({len(train_idx)/len(full_dataset)*100:.1f}%)")
print(f"  Val    : {len(val_idx):,} images   ({len(val_idx)/len(full_dataset)*100:.1f}%)")
print(f"  Test   : {len(test_idx):,} images   ({len(test_idx)/len(full_dataset)*100:.1f}%)")
print(f"  Total  : {len(train_idx)+len(val_idx)+len(test_idx):,} images")

# ── DataLoaders ───────────────────────────────────────────────────────────────
from torch.utils.data import Subset

train_dataset = Subset(FaceDataset(dataset_path, person_dirs, transform=train_transforms), train_idx)
val_dataset   = Subset(FaceDataset(dataset_path, person_dirs, transform=val_transforms),   val_idx)
test_dataset  = Subset(FaceDataset(dataset_path, person_dirs, transform=val_transforms),   test_idx)

# Train loader : PK Sampler
_train_base = FaceDataset(dataset_path, person_dirs, transform=train_transforms)
# Reindexer label_to_indices pour les indices du split train uniquement
train_label_to_indices = defaultdict(list)
for local_i, global_i in enumerate(train_idx):
    lbl = full_dataset.labels[global_i]
    train_label_to_indices[lbl].append(global_i)

class _SubsetPKSampler(Sampler):
    def __init__(self, label_to_indices, idx_map, P, K):
        # idx_map: global_idx -> position dans train_dataset
        self.label_to_indices = {
            lbl: [idx_map[gi] for gi in idxs if gi in idx_map]
            for lbl, idxs in label_to_indices.items()
        }
        self.label_to_indices = {k: v for k, v in self.label_to_indices.items() if len(v) >= 2}
        self.labels    = list(self.label_to_indices.keys())
        self.P, self.K = P, K
        self.n_batches = max(1, len(self.labels) // P)

    def __iter__(self):
        for _ in range(self.n_batches):
            batch_labels  = random.sample(self.labels, min(self.P, len(self.labels)))
            batch_indices = []
            for lbl in batch_labels:
                idxs = self.label_to_indices[lbl]
                chosen = random.choices(idxs, k=self.K) if len(idxs) < self.K else random.sample(idxs, self.K)
                batch_indices.extend(chosen)
            yield batch_indices

    def __len__(self):
        return self.n_batches

# Map: global index → local position in train_idx list
train_global_to_local = {gi: li for li, gi in enumerate(train_idx)}
pk_sampler = _SubsetPKSampler(train_label_to_indices, train_global_to_local, Config.P, Config.K)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=pk_sampler,
    num_workers=2,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print(f"Train loader : {len(train_loader)} batches/epoch (PK: P={Config.P}, K={Config.K})")
print(f"Val   loader : {len(val_loader)} batches")
print(f"Test  loader : {len(test_loader)} batches")

## 9. Optimiseur, Scheduler et Loss

**Phase 1** (5 epochs) : backbone gelé, seule la tête d'embedding est entraînée avec LR=1e-3.

**Phase 2** (35 epochs) : fine-tuning complet avec LR différentiel (backbone=1e-4, tête=1e-3) + CosineAnnealingLR.

In [ ]:
criterion = BatchHardTripletLoss(margin=Config.triplet_margin).to(device)

# Phase 1 : backbone gelé
model.freeze_backbone()
optimizer_phase1 = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=Config.lr_head,
    weight_decay=Config.weight_decay,
)

# Phase 2 : fine-tuning complet (sera créé au début de la phase 2)
# AdamW avec LR différentiel : backbone=1e-4, tête=1e-3

WARMUP_EPOCHS = 5   # Nombre d'epochs en Phase 1 (backbone gelé)

print(f"Loss           : BatchHardTripletLoss (margin={Config.triplet_margin})")
print(f"Phase 1        : {WARMUP_EPOCHS} epochs — backbone gelé — LR={Config.lr_head}")
print(f"Phase 2        : {Config.num_epochs - WARMUP_EPOCHS} epochs — fine-tuning — LR backbone={Config.lr_backbone} / tête={Config.lr_head}")
print(f"Scheduler      : CosineAnnealingLR (T_max = {Config.num_epochs - WARMUP_EPOCHS})")

## 10. Boucle d'Entraînement

Métrique principale pour early stopping : **val_loss** (triplet). On sauvegarde également l'AUC sur les paires de validation.

In [ ]:
def extract_embeddings(model, loader, device):
    """Extrait tous les embeddings et labels d'un DataLoader."""
    model.eval()
    all_emb, all_lbl = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Extracting", leave=False):
            imgs = imgs.to(device)
            emb  = model(imgs)
            all_emb.append(emb.cpu())
            all_lbl.append(labels)
    return torch.cat(all_emb), torch.cat(all_lbl)

def compute_val_auc(emb_np, lbl_np, n_pos=2000, n_neg=2000, seed=42):
    """Calcule l'AUC sur un sous-ensemble de paires de validation."""
    from sklearn.metrics import roc_auc_score
    rng = np.random.default_rng(seed)
    n   = len(emb_np)
    pos_scores, neg_scores = [], []

    # Paires positives
    label_to_idx = defaultdict(list)
    for i, l in enumerate(lbl_np):
        label_to_idx[int(l)].append(i)
    for _ in range(n_pos * 5):
        if len(pos_scores) >= n_pos: break
        lbl = int(rng.choice(list(label_to_idx.keys())))
        if len(label_to_idx[lbl]) < 2: continue
        i, j = rng.choice(label_to_idx[lbl], 2, replace=False)
        pos_scores.append(float(np.dot(emb_np[i], emb_np[j])))
    # Paires négatives
    for _ in range(n_neg * 3):
        if len(neg_scores) >= n_neg: break
        i, j = rng.integers(0, n, 2)
        if lbl_np[i] != lbl_np[j]:
            neg_scores.append(float(np.dot(emb_np[i], emb_np[j])))

    scores = np.array(pos_scores + neg_scores)
    labels = np.array([1]*len(pos_scores) + [0]*len(neg_scores))
    if len(np.unique(labels)) < 2: return 0.5
    return float(roc_auc_score(labels, scores))

# ── Training state ────────────────────────────────────────────────────────────
history = {
    'train_loss': [], 'val_loss': [], 'val_auc': [],
    'train_active': [], 'learning_rate': []
}
best_val_loss        = float('inf')
best_val_auc         = 0.0
early_stopping_counter = 0
best_model_state       = None
scheduler              = None
optimizer              = optimizer_phase1

print("Début de l'entraînement...")
print("=" * 70)

for epoch in range(Config.num_epochs):

    # ── Transition Phase 1 → Phase 2 ─────────────────────────────────────────
    if epoch == WARMUP_EPOCHS:
        print(f"\n{'='*70}")
        print(f"PHASE 2 — Fine-tuning complet (epoch {epoch+1})")
        print(f"{'='*70}")
        model.unfreeze_backbone()
        optimizer = AdamW(
            model.get_param_groups(Config.lr_head, Config.lr_backbone, Config.weight_decay)
        )
        scheduler = CosineAnnealingLR(
            optimizer,
            T_max   = Config.num_epochs - WARMUP_EPOCHS,
            eta_min = 1e-6,
        )

    print(f"\nEpoch [{epoch+1}/{Config.num_epochs}]  Phase={'1 (warmup)' if epoch < WARMUP_EPOCHS else '2 (fine-tune)'}")

    # ── TRAIN ─────────────────────────────────────────────────────────────────
    model.train()
    train_loss_sum, train_active_sum, n_batches = 0.0, 0.0, 0

    for imgs, labels in tqdm(train_loader, desc=f"  Train", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        embeddings       = model(imgs)
        loss, active_frac = criterion(embeddings, labels)
        if loss.item() > 0:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
        train_loss_sum   += loss.item()
        train_active_sum += active_frac
        n_batches        += 1
        del imgs, labels, embeddings
        if n_batches % 50 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if scheduler is not None:
        scheduler.step()

    train_loss   = train_loss_sum   / max(n_batches, 1)
    train_active = train_active_sum / max(n_batches, 1)

    # ── VALIDATION ────────────────────────────────────────────────────────────
    val_emb, val_lbl = extract_embeddings(model, val_loader, device)
    val_emb_np = val_emb.numpy()
    val_lbl_np = val_lbl.numpy()

    # Calcul val_loss en repassant sur quelques batches
    model.eval()
    val_loss_sum, val_n = 0.0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            emb  = model(imgs)
            loss, _ = criterion(emb, labels)
            val_loss_sum += loss.item()
            val_n += 1
    val_loss = val_loss_sum / max(val_n, 1)
    val_auc  = compute_val_auc(val_emb_np, val_lbl_np)

    current_lr = optimizer.param_groups[-1]['lr']
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    history['train_active'].append(train_active)
    history['learning_rate'].append(current_lr)

    print(f"  Train  loss={train_loss:.4f}  active_triplets={train_active*100:.1f}%")
    print(f"  Val    loss={val_loss:.4f}  AUC={val_auc:.4f}")
    print(f"  LR = {current_lr:.2e}")

    # ── Early stopping (sur val_loss) ──────────────────────────────────────────
    improved = val_loss < best_val_loss - Config.early_stopping_min_delta
    if improved:
        best_val_loss  = val_loss
        best_val_auc   = val_auc
        early_stopping_counter = 0
        best_model_state = {
            'model'    : {k: v.cpu() for k, v in model.state_dict().items()},
            'epoch'    : epoch,
            'val_loss' : val_loss,
            'val_auc'  : val_auc,
            'val_emb'  : val_emb,
            'val_lbl'  : val_lbl,
        }
        print(f"  ✓ Nouveau meilleur modèle  (val_loss={val_loss:.4f}, AUC={val_auc:.4f})")
    else:
        early_stopping_counter += 1
        print(f"  Pas d'amélioration ({early_stopping_counter}/{Config.early_stopping_patience})")
        if early_stopping_counter >= Config.early_stopping_patience:
            print(f"\nEarly stopping à l'epoch {epoch+1}")
            break

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 70)
print("ENTRAÎNEMENT TERMINÉ")
print(f"   Best val loss : {best_val_loss:.4f}")
print(f"   Best val AUC  : {best_val_auc:.4f}")
print(f"   Best epoch    : {best_model_state['epoch']+1}")
print("=" * 70)

## 11. Chargement du Meilleur Modèle et Évaluation Complète

Évaluation sur le **test set** avec le protocole 9 000 paires (6 000 positifs + 3 000 négatifs).

In [ ]:
if best_model_state is None:
    raise RuntimeError("Aucun modèle sauvegardé — vérifiez la boucle d'entraînement.")

model.load_state_dict(best_model_state['model'])
best_epoch = best_model_state['epoch']
print(f"Meilleur modèle chargé (epoch {best_epoch+1}, val_loss={best_model_state['val_loss']:.4f})")

# ── Extraire les embeddings du test set ───────────────────────────────────────
test_emb, test_lbl = extract_embeddings(model, test_loader, device)
test_emb_np = test_emb.numpy()
test_lbl_np = test_lbl.numpy()
print(f"Test embeddings : {test_emb_np.shape}")

# ── Construire les 9 000 paires de test ───────────────────────────────────────
rng = np.random.default_rng(Config.seed)
label_to_test_idx = defaultdict(list)
for i, lbl in enumerate(test_lbl_np):
    label_to_test_idx[int(lbl)].append(i)

pair_scores, pair_labels = [], []

# 6 000 paires positives
n_pos = 0
attempts = 0
while n_pos < Config.test_pos_pairs and attempts < Config.test_pos_pairs * 20:
    attempts += 1
    lbl = int(rng.choice(list(label_to_test_idx.keys())))
    if len(label_to_test_idx[lbl]) < 2:
        continue
    i, j = rng.choice(label_to_test_idx[lbl], 2, replace=False)
    sim = float(np.dot(test_emb_np[i], test_emb_np[j]))
    pair_scores.append(sim)
    pair_labels.append(1)
    n_pos += 1

# 3 000 paires négatives
n_neg = 0
attempts = 0
while n_neg < Config.test_neg_pairs and attempts < Config.test_neg_pairs * 20:
    attempts += 1
    i, j = rng.integers(0, len(test_emb_np), 2)
    if test_lbl_np[i] != test_lbl_np[j]:
        sim = float(np.dot(test_emb_np[i], test_emb_np[j]))
        pair_scores.append(sim)
        pair_labels.append(0)
        n_neg += 1

pair_scores = np.array(pair_scores, dtype=np.float32)
pair_labels = np.array(pair_labels, dtype=np.int32)
print(f"Paires test : {len(pair_scores):,}  (positifs={pair_labels.sum():,}, négatifs={(1-pair_labels).sum():,})")

# ── ROC / AUC ─────────────────────────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(pair_labels, pair_scores)
auc_score = auc(fpr, tpr)
far_arr, frr_arr, tar_arr = fpr, 1.0 - tpr, tpr

# ── EER ───────────────────────────────────────────────────────────────────────
eer_idx       = np.argmin(np.abs(far_arr - frr_arr))
eer_value     = float((far_arr[eer_idx] + frr_arr[eer_idx]) / 2.0)
eer_threshold = float(thresholds[eer_idx])
tar_at_eer    = float(tar_arr[eer_idx])
far_at_eer    = float(far_arr[eer_idx])

# ── TAR @ FAR fixed ───────────────────────────────────────────────────────────
def tar_at_far_val(target_far):
    idx = np.searchsorted(far_arr, target_far, side='right') - 1
    return float(tar_arr[int(np.clip(idx, 0, len(tar_arr)-1))])

tar_far_0001 = tar_at_far_val(0.0001)
tar_far_001  = tar_at_far_val(0.001)
tar_far_01   = tar_at_far_val(0.01)
tar_far_10   = tar_at_far_val(0.1)

# ── d-prime ───────────────────────────────────────────────────────────────────
genuine_scores  = pair_scores[pair_labels == 1]
impostor_scores = pair_scores[pair_labels == 0]
mu_g, std_g     = genuine_scores.mean(), genuine_scores.std()
mu_i, std_i     = impostor_scores.mean(), impostor_scores.std()
d_prime         = abs(mu_g - mu_i) / (np.sqrt(0.5 * (std_g**2 + std_i**2)) + 1e-8)

# ── Optimal threshold ─────────────────────────────────────────────────────────
opt_idx          = np.argmin(far_arr + frr_arr)
optimal_threshold = float(thresholds[opt_idx])
false_accepts     = float(far_arr[opt_idx])
false_rejects     = float(frr_arr[opt_idx])

print("\n" + "=" * 70)
print("MÉTRIQUES D'ÉVALUATION — TEST SET")
print("=" * 70)
print(f"\n  ROC / Vérification")
print(f"    AUC                  : {auc_score:.4f}")
print(f"    d-prime              : {d_prime:.4f}")
print(f"\n  EER")
print(f"    EER                  : {eer_value*100:.2f}%")
print(f"    EER threshold        : {eer_threshold:.4f}")
print(f"    TAR @ EER threshold  : {tar_at_eer*100:.2f}%")
print(f"    FAR @ EER threshold  : {far_at_eer*100:.2f}%")
print(f"\n  TAR @ FAR fixé")
print(f"    TAR @ FAR=0.0001     : {tar_far_0001*100:.2f}%")
print(f"    TAR @ FAR=0.001      : {tar_far_001*100:.2f}%")
print(f"    TAR @ FAR=0.01       : {tar_far_01*100:.2f}%")
print(f"    TAR @ FAR=0.1        : {tar_far_10*100:.2f}%")
print(f"\n  Point de fonctionnement optimal (min FAR+FRR)")
print(f"    Threshold            : {optimal_threshold:.4f}")
print(f"    FAR                  : {false_accepts*100:.2f}%")
print(f"    FRR                  : {false_rejects*100:.2f}%")
print(f"\n  Distributions de scores")
print(f"    Genuine  : mu={mu_g:.4f}  std={std_g:.4f}")
print(f"    Impostor : mu={mu_i:.4f}  std={std_i:.4f}")
print("=" * 70)

## 12. Visualisations

In [ ]:
out = Config.output_dir
os.makedirs(out, exist_ok=True)

# ── 1. Training curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('Courbes d\'entraînement — MobileNetV3 + Batch Hard Triplet Loss', fontsize=14, fontweight='bold')
epochs_done = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_done, history['train_loss'], label='Train', lw=2)
axes[0].plot(epochs_done, history['val_loss'],   label='Val',   lw=2)
axes[0].axvline(x=WARMUP_EPOCHS, color='red', linestyle='--', alpha=0.6, label='Ph1→Ph2')
axes[0].set_title('Triplet Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_done, history['val_auc'], color='purple', lw=2, label='Val AUC')
axes[1].set_ylim([0.5, 1.05])
axes[1].set_title('AUC Validation'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs_done, [x*100 for x in history['train_active']], color='orange', lw=2)
axes[2].set_title('Triplets actifs (%)'); axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('%'); axes[2].grid(alpha=0.3)

axes[3].plot(epochs_done, history['learning_rate'], color='green', lw=2)
axes[3].set_title('Learning Rate'); axes[3].set_xlabel('Epoch'); axes[3].set_yscale('log'); axes[3].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{out}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("training_curves.png saved")

# ── 2. ROC Curve ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ax.plot(fpr, tpr, lw=2.5, label=f'ROC (AUC={auc_score:.4f})')
ax.plot([0,1],[0,1],'k--',lw=1,label='Random')
ax.scatter(far_at_eer, tar_at_eer, color='red', s=120, zorder=5, label=f'EER={eer_value*100:.2f}%')
for far_op,tar_op,col,lbl in [(0.0001,tar_far_0001,'green','FAR=0.01%'),(0.001,tar_far_001,'orange','FAR=0.1%'),(0.01,tar_far_01,'purple','FAR=1%')]:
    ax.scatter(far_op,tar_op,s=80,zorder=5,color=col,label=f'TAR={tar_op*100:.1f}% @ {lbl}')
ax.set_xlabel('FAR'); ax.set_ylabel('TAR'); ax.set_title('Courbe ROC'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
valid = fpr > 0
ax.semilogx(fpr[valid], tpr[valid], lw=2.5, label=f'AUC={auc_score:.4f}')
ax.scatter(far_at_eer, tar_at_eer, color='red', s=120, zorder=5, label=f'EER={eer_value*100:.2f}%')
for far_op,tar_op,col,lbl in [(0.0001,tar_far_0001,'green','0.0001'),(0.001,tar_far_001,'orange','0.001'),(0.01,tar_far_01,'purple','0.01'),(0.1,tar_far_10,'brown','0.1')]:
    ax.axvline(x=far_op, color=col, linestyle=':', alpha=0.6)
    ax.scatter(far_op, tar_op, s=80, zorder=5, color=col, label=f'TAR={tar_op*100:.1f}%@FAR={lbl}')
ax.set_xlabel('FAR (log)'); ax.set_ylabel('TAR'); ax.set_title('ROC (Log FAR)'); ax.legend(fontsize=7); ax.grid(alpha=0.3)
ax.set_xlim([1e-5,1.0]); ax.set_ylim([0,1.05])

plt.tight_layout()
plt.savefig(f'{out}/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("roc_curve.png saved")

# ── 3. DET Curve ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8,7))
valid = (far_arr>0)&(frr_arr>0)
ax.loglog(far_arr[valid]*100, frr_arr[valid]*100, lw=2.5)
ax.scatter(eer_value*100, eer_value*100, color='red', s=150, zorder=5, label=f'EER={eer_value*100:.2f}%')
ax.set_xlabel('FAR (%)'); ax.set_ylabel('FRR (%)'); ax.set_title('Courbe DET')
ax.legend(); ax.grid(alpha=0.3, which='both')
plt.tight_layout(); plt.savefig(f'{out}/det_curve.png', dpi=150, bbox_inches='tight'); plt.show()
print("det_curve.png saved")

# ── 4. Score distributions ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10,5))
ax.hist(impostor_scores, bins=100, alpha=0.6, color='red',   label='Impostor', density=True)
ax.hist(genuine_scores,  bins=100, alpha=0.6, color='green', label='Genuine',  density=True)
ax.axvline(eer_threshold,      color='black', linestyle='--', lw=2, label=f'EER thr={eer_threshold:.4f}')
ax.axvline(optimal_threshold,  color='blue',  linestyle=':',  lw=2, label=f'Opt thr={optimal_threshold:.4f}')
ax.set_xlabel('Cosine Similarity'); ax.set_ylabel('Density')
ax.set_title(f'Score Distributions  (d-prime = {d_prime:.3f})')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{out}/score_distributions.png', dpi=150, bbox_inches='tight'); plt.show()
print("score_distributions.png saved")

# ── 5. t-SNE ──────────────────────────────────────────────────────────────────
print("\nCalcul t-SNE (peut prendre quelques minutes)...")
val_emb_np = best_model_state['val_emb'].numpy()
val_lbl_np = best_model_state['val_lbl'].numpy()
max_tsne   = min(2000, len(val_emb_np))
idx_tsne   = np.random.choice(len(val_emb_np), max_tsne, replace=False)
tsne       = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
emb_2d     = tsne.fit_transform(val_emb_np[idx_tsne])
fig, ax    = plt.subplots(figsize=(12,9))
sc = ax.scatter(emb_2d[:,0], emb_2d[:,1], c=val_lbl_np[idx_tsne], cmap='tab20', s=20, alpha=0.7)
ax.set_title('t-SNE — Embeddings validation (MobileNetV3-Large 512D)')
ax.set_xlabel('Dim 1'); ax.set_ylabel('Dim 2')
plt.colorbar(sc, ax=ax, label='Class ID')
plt.tight_layout(); plt.savefig(f'{out}/tsne_embeddings.png', dpi=150, bbox_inches='tight'); plt.show()
print("tsne_embeddings.png saved")

## 13. Tableau de Résultats du Modèle

In [ ]:
# ── Tableau de résultats complet ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
ax.axis('off')

table_data = [
    ['Catégorie', 'Métrique', 'Valeur'],
    # Architecture
    ['Architecture', 'Backbone',         'MobileNetV3-Large'],
    ['Architecture', 'Embedding dim',    '512D (L2-normalisé)'],
    ['Architecture', 'Loss',             f'Batch Hard Triplet (margin={Config.triplet_margin})'],
    ['Architecture', 'Image size',       f'{Config.image_size}×{Config.image_size}'],
    # Dataset & Split
    ['Dataset',      'Personnes',         str(Config.num_classes)],
    ['Dataset',      'Images totales',    f'{len(full_dataset):,}'],
    ['Dataset',      'Train',             f'{len(train_idx):,} images (80%)'],
    ['Dataset',      'Val',               f'{len(val_idx):,} images (10%)'],
    ['Dataset',      'Test',              f'{len(test_idx):,} images (10%)'],
    # Training
    ['Entraînement', 'Epochs (best)',     f'{best_epoch+1} / {Config.num_epochs}'],
    ['Entraînement', 'Batch (P×K)',       f'{Config.P}×{Config.K} = {Config.batch_size}'],
    ['Entraînement', 'LR backbone',       str(Config.lr_backbone)],
    ['Entraînement', 'LR tête',           str(Config.lr_head)],
    # Test metrics
    ['Test (9 000 paires)', 'AUC',                  f'{auc_score:.4f}'],
    ['Test (9 000 paires)', 'd-prime',               f'{d_prime:.4f}'],
    ['Test (9 000 paires)', 'EER',                   f'{eer_value*100:.2f}%'],
    ['Test (9 000 paires)', 'EER threshold',         f'{eer_threshold:.4f}'],
    ['Test (9 000 paires)', 'TAR @ EER',             f'{tar_at_eer*100:.2f}%'],
    ['Test (9 000 paires)', 'TAR @ FAR=0.0001',      f'{tar_far_0001*100:.2f}%'],
    ['Test (9 000 paires)', 'TAR @ FAR=0.001',       f'{tar_far_001*100:.2f}%'],
    ['Test (9 000 paires)', 'TAR @ FAR=0.01',        f'{tar_far_01*100:.2f}%'],
    ['Test (9 000 paires)', 'TAR @ FAR=0.1',         f'{tar_far_10*100:.2f}%'],
    ['Test (9 000 paires)', 'Optimal FAR',            f'{false_accepts*100:.2f}%'],
    ['Test (9 000 paires)', 'Optimal FRR',            f'{false_rejects*100:.2f}%'],
    ['Test (9 000 paires)', 'Optimal threshold',      f'{optimal_threshold:.4f}'],
    ['Distributions',       'Genuine (mu / std)',    f'{mu_g:.4f} / {std_g:.4f}'],
    ['Distributions',       'Impostor (mu / std)',   f'{mu_i:.4f} / {std_i:.4f}'],
]

# Couleurs par catégorie
cat_colors = {
    'Architecture'      : '#2980b9',
    'Dataset'           : '#27ae60',
    'Entraînement'      : '#8e44ad',
    'Test (9 000 paires)': '#c0392b',
    'Distributions'     : '#d35400',
}

tbl = ax.table(
    cellText  = [[r[1], r[2]] for r in table_data[1:]],
    colLabels = ['Métrique', 'Valeur'],
    rowLabels = [r[0] for r in table_data[1:]],
    cellLoc   = 'left',
    rowLoc    = 'left',
    loc       = 'center',
    bbox      = [0.0, 0.0, 1.0, 1.0],
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)

# Style header
for j in range(2):
    tbl[(0, j)].set_facecolor('#1a1a2e')
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')

# Style rows
prev_cat = None
for i, row in enumerate(table_data[1:], 1):
    cat   = row[0]
    color = cat_colors.get(cat, '#ecf0f1')
    alpha_bg = '#f8f9fa' if i % 2 == 0 else '#ffffff'
    # Row label cell
    tbl[(i, -1)].set_facecolor(color)
    tbl[(i, -1)].set_text_props(color='white', fontweight='bold', fontsize=7)
    for j in range(2):
        tbl[(i, j)].set_facecolor(alpha_bg)

ax.set_title(
    f'Résultats — MobileNetV3-Large + Batch Hard Triplet Loss\n'
    f'Split 80/10/10 | Image 112×112 | Embedding 512D',
    fontsize=12, fontweight='bold', pad=20
)
plt.tight_layout()
plt.savefig(f'{out}/results_table.png', dpi=150, bbox_inches='tight')
plt.show()
print("results_table.png saved")

## 14. Sauvegarde du Modèle

In [ ]:
out = Config.output_dir
os.makedirs(out, exist_ok=True)

save_dict = {
    'model_state_dict' : best_model_state['model'],
    'class_to_idx'     : full_dataset.class_to_idx,
    'idx_to_class'     : full_dataset.idx_to_class,
    'config': {
        'image_size'      : Config.image_size,
        'embedding_dim'   : Config.embedding_dim,
        'num_classes'     : Config.num_classes,
        'triplet_margin'  : Config.triplet_margin,
        'model_name'      : Config.model_name,
        'seed'            : Config.seed,
    },
    'metrics': {
        'best_epoch'        : best_epoch,
        'val_loss'          : float(best_model_state['val_loss']),
        'val_auc'           : float(best_model_state['val_auc']),
        'test_auc'          : float(auc_score),
        'd_prime'           : float(d_prime),
        'eer'               : float(eer_value),
        'eer_threshold'     : float(eer_threshold),
        'tar_at_eer'        : float(tar_at_eer),
        'far_at_eer'        : float(far_at_eer),
        'tar_at_far_0001'   : float(tar_far_0001),
        'tar_at_far_001'    : float(tar_far_001),
        'tar_at_far_01'     : float(tar_far_01),
        'tar_at_far_10'     : float(tar_far_10),
        'optimal_threshold' : float(optimal_threshold),
        'far'               : float(false_accepts),
        'frr'               : float(false_rejects),
        'genuine_mean'      : float(mu_g),
        'genuine_std'       : float(std_g),
        'impostor_mean'     : float(mu_i),
        'impostor_std'      : float(std_i),
    },
    'history'  : {k: [float(v) for v in vals] for k, vals in history.items()},
    'split_info': {
        'train_idx': train_idx,
        'val_idx'  : val_idx,
        'test_idx' : test_idx,
    },
}

ckpt_path = f'{out}/best_triplet_model.pth'
torch.save(save_dict, ckpt_path)
print(f"Checkpoint complet   : {ckpt_path}  ({os.path.getsize(ckpt_path)/1e6:.1f} MB)")

w_path = f'{out}/best_triplet_weights_only.pth'
torch.save(best_model_state['model'], w_path)
print(f"Weights seulement    : {w_path}  ({os.path.getsize(w_path)/1e6:.1f} MB)")

for name, data in [
    ('model_config',  save_dict['config']),
    ('model_metrics', save_dict['metrics']),
    ('class_to_idx',  full_dataset.class_to_idx),
]:
    p = f'{out}/{name}.json'
    with open(p, 'w') as f:
        json.dump(data, f, indent=2)
    print(f"{name}.json saved")

## 15. Résumé Final

In [ ]:
print("\n" + "=" * 80)
print(" " * 25 + "RÉSUMÉ FINAL — FACE VERIFICATION")
print("=" * 80)

print(f"\nARCHITECTURE")
print(f"   Backbone     : MobileNetV3-Large (ImageNet pre-trained)")
print(f"   Image size   : {Config.image_size}×{Config.image_size}")
print(f"   Embedding    : 512D (L2-normalized)")
print(f"   Loss         : Batch Hard Triplet Loss (margin={Config.triplet_margin})")

print(f"\nDATASET  (Split 80/10/10)")
print(f"   Personnes    : {Config.num_classes}")
print(f"   Train        : {len(train_idx):,} images")
print(f"   Val          : {len(val_idx):,} images")
print(f"   Test         : {len(test_idx):,} images")

print(f"\nENTRAÎNEMENT")
print(f"   Batch (P×K)  : {Config.P}×{Config.K} = {Config.batch_size}")
print(f"   Epochs       : {len(history['train_loss'])}/{Config.num_epochs}  (best={best_epoch+1})")
print(f"   LR backbone  : {Config.lr_backbone}  →  tête : {Config.lr_head}")
print(f"   Scheduler    : CosineAnnealingLR")

print(f"\nMÉTRIQUES TEST (9 000 paires)")
print(f"   AUC          : {auc_score:.4f}")
print(f"   d-prime      : {d_prime:.4f}")
print(f"   EER          : {eer_value*100:.2f}%  (thr={eer_threshold:.4f})")
print(f"   TAR @ FAR=0.001 : {tar_far_001*100:.2f}%")
print(f"   TAR @ FAR=0.01  : {tar_far_01*100:.2f}%")
print(f"   Optimal FAR/FRR : {false_accepts*100:.2f}% / {false_rejects*100:.2f}%")

print(f"\nFICHIERS  →  {Config.output_dir}/")
for fname in ['best_triplet_model.pth', 'best_triplet_weights_only.pth',
              'model_config.json', 'model_metrics.json', 'class_to_idx.json',
              'training_curves.png', 'roc_curve.png', 'det_curve.png',
              'score_distributions.png', 'tsne_embeddings.png', 'results_table.png']:
    p      = f"{Config.output_dir}/{fname}"
    status = "✓ OK" if os.path.exists(p) else "✗ MISSING"
    print(f"   [{status}] {fname}")

print("\n" + "=" * 80)
print(" " * 20 + "ENTRAÎNEMENT COMPLET — MODÈLE PRÊT")
print("=" * 80)

## 16. Test sur une Image Individuelle (Inférence)

Ce bloc est **autonome** : chargez le checkpoint et testez n'importe quelle image.
Deux modes disponibles :
- **Identification** : retourne l'identité la plus proche parmi les personnes connues
- **Vérification** : compare deux images et décide si c'est la même personne

In [ ]:
import os, torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
from torchvision import transforms
from PIL import Image
import timm, json

# =====================================================================
# CLASSES (copied to keep the block standalone)
# =====================================================================

class _FaceVerificationModel(nn.Module):
    def __init__(self, embedding_dim=512, model_name="mobilenetv3_large_100", image_size=112):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=False,
            num_classes=0, global_pool='avg'
        )

        # Infer actual backbone output size (timm variants can differ)
        was_training = self.backbone.training
        self.backbone.eval()
        with torch.no_grad():
            dummy = torch.zeros(1, 3, image_size, image_size)
            backbone_out = self.backbone(dummy).shape[1]
        if was_training:
            self.backbone.train()

        self.embedding_head = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim, bias=False),
            nn.BatchNorm1d(embedding_dim),
        )

    def forward(self, x):
        return F.normalize(self.embedding_head(self.backbone(x)), p=2, dim=1)


# =====================================================================
# CHARGEMENT DU MODELE
# =====================================================================

def load_face_model(checkpoint_path, device='cpu'):
    """Charge le modele et retourne (model, cfg, idx_to_class, optimal_threshold)."""
    ckpt         = torch.load(checkpoint_path, map_location=device, weights_only=False)
    cfg          = ckpt['config']
    idx_to_class = {int(k): v for k, v in ckpt.get('idx_to_class', {}).items()}
    thresh       = ckpt['metrics']['optimal_threshold']

    mdl = _FaceVerificationModel(
        cfg['embedding_dim'],
        model_name=cfg.get('model_name', 'mobilenetv3_large_100'),
        image_size=cfg.get('image_size', 112),
    ).to(device)
    mdl.load_state_dict(ckpt['model_state_dict'])
    mdl.eval()
    return mdl, cfg, idx_to_class, thresh


# =====================================================================
# PREPROCESSING
# =====================================================================

def _get_transform(image_size):
    return transforms.Compose([
        transforms.Resize((image_size + 20, image_size + 20)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

def get_embedding(model, image_path, image_size=112, device='cpu'):
    """Retourne l'embedding (1, 512) d'une image."""
    tfm    = _get_transform(image_size)
    img    = Image.open(image_path).convert('RGB')
    tensor = tfm(img).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = model(tensor)
    return emb.cpu()


# =====================================================================
# MODE 1 : IDENTIFICATION  (1 image -> identite predite)
# =====================================================================

def identify_person(model, cfg, idx_to_class, image_path, top_k=5, device='cpu'):
    """
    Identifie la personne la plus proche dans l'espace d'embedding.

    Retourne :
        predicted_identity (str)  - nom du dossier le plus proche
        top_k_results (list)      - [(identity, cosine_sim), ...]
    """
    # Recuperer les prototypes (barycentres des classes) depuis le modele entraine
    # Si non disponible, on utilise les logits du dernier checkpoint
    emb = get_embedding(model, image_path, cfg['image_size'], device)  # (1, 512)

    # Sans galerie explicite on ne peut pas faire d'identification classique.
    # Ce bloc suppose qu'une galerie est disponible ou utilise les embeddings de val.
    print("[INFO] Pour l'identification sans galerie, cf. Mode 2 (verification 1-vs-1).")
    print(f"       Embedding shape : {emb.shape}  norme={emb.norm().item():.4f}")
    return None, []


# =====================================================================
# MODE 2 : VERIFICATION  (2 images -> meme personne ?)
# =====================================================================

def verify_faces(model, cfg, threshold, image_path_1, image_path_2, device='cpu'):
    """
    Compare deux images et decide si elles appartiennent a la meme personne.

    Retourne :
        same_person  (bool)   - True si cosine_sim > threshold
        similarity   (float)  - score de similarite cosinus in [-1, 1]
        decision     (str)    - 'MEME PERSONNE' ou 'PERSONNES DIFFERENTES'
    """
    emb1 = get_embedding(model, image_path_1, cfg['image_size'], device)
    emb2 = get_embedding(model, image_path_2, cfg['image_size'], device)

    similarity  = float(F.cosine_similarity(emb1, emb2).item())
    same_person = similarity >= threshold
    decision    = 'MEME PERSONNE' if same_person else 'PERSONNES DIFFERENTES'

    print("=" * 60)
    print(f"  Image 1     : {os.path.basename(image_path_1)}")
    print(f"  Image 2     : {os.path.basename(image_path_2)}")
    print(f"  Similarite  : {similarity:.4f}")
    print(f"  Threshold   : {threshold:.4f}")
    print(f"  Decision    : {decision}")
    print("=" * 60)
    return same_person, similarity, decision


# =====================================================================
# MODE 3 : TEST AVEC GALERIE  (embedding query vs galerie de reference)
# =====================================================================

def identify_from_gallery(model, cfg, idx_to_class, threshold,
                           query_image_path, gallery_embeddings, gallery_labels,
                           top_k=3, device='cpu'):
    """
    Identifie une personne par rapport a une galerie prealablement extraite.

    Args:
        gallery_embeddings : np.ndarray (N, 512) - embeddings de reference
        gallery_labels     : np.ndarray (N,)     - labels correspondants

    Retourne :
        predicted_identity  (str)
        confidence          (float)  - score cosinus de la meilleure correspondance
        accepted            (bool)   - True si confidence > threshold
        top_k_results       (list)   - [(identity, sim), ...]
    """
    emb_q = get_embedding(model, query_image_path, cfg['image_size'], device).numpy()  # (1, 512)
    sims  = (gallery_embeddings @ emb_q.T).squeeze()  # cosine sim (galerie deja L2-norm)

    top_k_idx     = np.argsort(sims)[::-1][:top_k]
    top_k_results = [(idx_to_class.get(int(gallery_labels[i]), f'cls_{gallery_labels[i]}'),
                      float(sims[i])) for i in top_k_idx]

    best_label, best_sim = top_k_results[0]
    accepted = best_sim >= threshold

    print("=" * 60)
    print(f"  Query image : {os.path.basename(query_image_path)}")
    print(f"  Top-{top_k} resultats :")
    for rank, (ident, sim) in enumerate(top_k_results, 1):
        marker = "*" if rank == 1 else " "
        print(f"    {marker} #{rank}: {ident:<30}  sim={sim:.4f}")
    print(f"  Decision    : {'ACCEPTE - ' + best_label if accepted else 'REJETE (inconnu)'}")
    print(f"  Threshold   : {threshold:.4f}")
    print("=" * 60)
    return best_label, best_sim, accepted, top_k_results


# =====================================================================
# DEMONSTRATION - MODIFIEZ LES CHEMINS CI-DESSOUS
# =====================================================================

CKPT  = '/kaggle/working/best_triplet_model.pth'
IMG_1 = 'path/to/image1.jpg'   # Remplacez par un vrai chemin
IMG_2 = 'path/to/image2.jpg'   # Meme personne ou personne differente

_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if os.path.exists(CKPT):
    _model, _cfg, _idx_to_class, _threshold = load_face_model(CKPT, device=str(_device))
    print(f"Modele charge depuis {CKPT}")
    print(f"  Classes          : {_cfg['num_classes']}")
    print(f"  Embedding dim    : {_cfg['embedding_dim']}")
    print(f"  Optimal threshold: {_threshold:.4f}")
    print(f"  Exemple classe 0 : {_idx_to_class.get(0, 'N/A')}")

    # Test de verification (2 images)
    if os.path.exists(IMG_1) and os.path.exists(IMG_2):
        same, sim, decision = verify_faces(_model, _cfg, _threshold, IMG_1, IMG_2, device=str(_device))
    else:
        print("\n[DEMO] Definissez IMG_1 et IMG_2 pour lancer le test de verification.")
        print(f"       IMG_1 = '{IMG_1}'")
        print(f"       IMG_2 = '{IMG_2}'")

    # Test rapide : embedding d'une image
    if os.path.exists(IMG_1):
        emb_test = get_embedding(_model, IMG_1, _cfg['image_size'], device=str(_device))
        print(f"\nEmbedding de {os.path.basename(IMG_1)} :")
        print(f"  Shape : {emb_test.shape}")
        print(f"  Norme : {emb_test.norm().item():.6f}  (doit etre ~ 1.0)")
        print(f"  Min / Max / Mean : {emb_test.min().item():.4f} / {emb_test.max().item():.4f} / {emb_test.mean().item():.4f}")
else:
    print(f"[INFO] Checkpoint non trouve : {CKPT}")
    print("       Executez d'abord les cellules d'entrainement.")

## Récapitulatif des Fichiers de Sortie

| Fichier | Description |
|---------|-------------|
| `best_triplet_model.pth` | Checkpoint complet (modèle + métriques + split) |
| `best_triplet_weights_only.pth` | Poids seuls (plus léger) |
| `model_config.json` | Hyperparamètres du modèle |
| `model_metrics.json` | Toutes les métriques de test |
| `class_to_idx.json` | Mapping nom_dossier ↔ index |
| `training_curves.png` | Loss, AUC, triplets actifs, LR |
| `roc_curve.png` | ROC standard + ROC log-FAR |
| `det_curve.png` | Courbe DET |
| `score_distributions.png` | Genuine vs Impostor |
| `tsne_embeddings.png` | t-SNE des embeddings de validation |
| `results_table.png` | Tableau complet des résultats |